# TKCE — clean-dataset honest benchmark

**Where we are:** de-leaked `eye_movements` showed the famous tree-vs-NN gap there was almost entirely the data leak (ceiling 0.708 -> 0.554; gap +0.125 -> +0.011 ~ noise). That dataset is now a *leakage case study*, not a benchmark. The method's real test happens on **clean** Grinsztajn datasets where trees beat raw NNs legitimately.

**This notebook:** 4 clean numeric-classification datasets x {raw NN, x+tree infold, x+tree OOB-honest}, 3-member ensembles, 400 epochs.

| task | dataset |
|---|---|
| 361062 | pol |
| 361063 | house_16H |
| 361065 | MagicTelescope |
| 361277 | california |

### How to run
1. Runtime -> **GPU** (A100).
2. **Run all** (~60-90 min; to shorten, delete datasets from the list in cell 4).

### What to look for, per dataset
- **tree ceiling vs raw NN** = the genuine tree advantage (no leak inflating it)
- does **x+tree** close that gap honestly?
- **infold vs oob**: does the naive encoding overstate itself (train AUC 1.0, early best-epoch) while OOB gives cleaner curves / equal-or-better test?

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> set Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 4 · THE BENCHMARK MATRIX  (per dataset: raw once, then x+tree infold vs oob)
TASKS = {361062: 'pol', 361063: 'house_16H', 361065: 'MagicTelescope', 361277: 'california'}
reg = ('--ensemble 3 --epochs 400 --dropout 0.3 --l1 5e-5 '
       '--weight-decay 1e-3 --lr 3e-4 --device auto')
for tid, name in TASKS.items():
    print(f'\n############################  {name} ({tid})  ############################')
    !python -u run_fusion.py --task {tid} {reg} --views 'x'      --encoding infold --out results/fusion/{name}_raw
    !python -u run_fusion.py --task {tid} {reg} --views 'x+tree' --encoding infold --out results/fusion/{name}_infold
    !python -u run_fusion.py --task {tid} {reg} --views 'x+tree' --encoding oob    --out results/fusion/{name}_oob

In [ ]:
# 5 · summary table across all runs
import json, glob
rows = []
for f in sorted(glob.glob('results/fusion/*/fusion_*.json')):
    s = json.load(open(f))
    tag = f.split('/')[2]
    for r in s['results']:
        rows.append((s['dataset'], tag.split('_')[-1], r['model'], r['test_auc'],
                     s['tree_ceiling'], round(r['test_auc'] - s['tree_ceiling'], 4)))
print(f"{'dataset':16s} {'run':8s} {'model':14s} {'test_auc':>8s} {'ceiling':>8s} {'vs_ceil':>8s}")
for d, tag, m, a, c, gap in rows:
    print(f'{d:16s} {tag:8s} {m:14s} {a:8.4f} {c:8.4f} {gap:+8.4f}')

In [ ]:
# 6 · show all figures
from IPython.display import Image, display
import glob, os
for f in sorted(glob.glob('results/fusion/*/*.png')):
    print('==', f, '==')
    display(Image(f))

In [ ]:
# 7 · download everything as one zip
import shutil
from google.colab import files
shutil.make_archive('fusion_clean_benchmark', 'zip', 'results/fusion')
files.download('fusion_clean_benchmark.zip')